# Jarvis: make my own voice (free)

This notebook turns a recording of **your voice** into a voice file that Jarvis uses offline on your phone.

**Before you start**
1. In Google Drive, make a folder named **Jarvis** (in "My Drive").
2. Put **one** of these in it:
   - **Best:** a recording of 20+ minutes of only your voice (any topic, English, quiet room). `.mp3`, `.m4a`, `.wav` all work. Several files are fine.
   - **Or** `jarvis-voice-data.zip` exported from Jarvis setup > Record my voice.
   - **Or (quick, lower quality):** a 1-minute recording named `short-clip` (for example `short-clip.m4a`).
3. In the menu: **Runtime > Change runtime type > T4 GPU > Save**.

**Then:** press **Runtime > Run all**. Allow Google Drive when asked. It takes about 3 to 5 hours.
If Colab stops (it can after a few hours), open the notebook again and press **Run all** again: it continues from where it stopped.

At the end you get **jarvis-my-voice.zip** in your Drive's Jarvis folder. Download it to your phone, then in Jarvis setup tap **Use my voice**.

Your recordings stay in your own Google Drive.

In [ ]:
#@title 1. Settings and Google Drive
EXTRA_EPOCHS = 400   # more = closer to your voice but longer. 300-600 is a good range.
JARVIS = "/content/drive/MyDrive/Jarvis"

import os, subprocess, glob, shutil, json, time, pathlib
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(JARVIS, exist_ok=True)
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpu or "NO GPU! Use Runtime > Change runtime type > T4 GPU, then Run all again.")
assert gpu, "A GPU is needed."
print("Files in your Jarvis folder:", os.listdir(JARVIS))

In [ ]:
#@title 2. Install the free tools (about 5 minutes)
PIPER_COMMIT = "d76f335d9eb5dea5ef6d7d70b2d6114e6303fa08"  # piper1-gpl, fixed version
!apt-get -qq update && apt-get -qq install -y build-essential cmake ninja-build ffmpeg > /dev/null
if not os.path.exists("/content/piper1-gpl"):
    !git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
    !cd /content/piper1-gpl && git checkout -q {PIPER_COMMIT}
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python setup.py build_ext --inplace > /dev/null
!pip -q install faster-whisper==1.1.1 onnx sherpa-onnx==1.12.33 soundfile librosa
print("Tools ready.")

In [ ]:
#@title 3. Prepare your recordings (writes the text automatically)
import soundfile as sf, librosa, numpy as np
DATA = "/content/data"; WAVS = DATA + "/wavs"
os.makedirs(WAVS, exist_ok=True)
rows = []
files = os.listdir(JARVIS)
AUDIO = (".mp3", ".m4a", ".wav", ".ogg", ".opus", ".aac", ".flac", ".mp4", ".amr", ".3gp", ".webm")
SENTENCES = ["Hello Boss, how can I help you today?", "Okay Boss, opening WhatsApp now.", "The weather in Hyderabad is warm and sunny this afternoon.", "Your battery is at forty five percent and charging.", "I have set an alarm for six thirty tomorrow morning.", "Sorry Boss, I didn't catch that. Please say it again.", "You have three new messages and one missed call.", "Turning on the torch for you.", "It's seven fifteen in the evening on a quiet Sunday.", "Should I send this message to Ravi?", "Calling your brother now. Please wait a moment.", "The meeting starts at ten o'clock in the main hall.", "Please remember to drink water and take a short break.", "Searching YouTube for the latest Telugu movie songs.", "Volume is now at eighty percent.", "I couldn't find that contact in your phone.", "Good morning! It looks like a bright and pleasant day.", "Here is what I found about the cricket match yesterday.", "Your next alarm is at five forty five in the morning.", "Do Not Disturb is on. I will keep things quiet.", "Internet is unavailable, but I can still do offline tasks.", "The quick brown fox jumps over the lazy dog near the river bank.", "A big black bug bit a bold brown bear by the bridge.", "She sells fresh vegetables at the market every Thursday.", "We watched the sunrise from the top of the green hill.", "My grandmother tells the best stories about her village.", "Please close the window before the rain comes in.", "The train to Vijayawada leaves at nine twenty from platform four.", "He ordered hot dosa, idli, and a cup of strong filter coffee.", "The children laughed loudly while playing in the park.", "Thirty two thousand people watched the final match.", "Could you pass me the blue notebook on the table?", "The library opens early on weekdays and late on Saturdays.", "Light travels much faster than sound through the air.", "I forgot my umbrella at the bus stop this morning.", "Our new office is on the fifth floor of the tall building.", "Every journey begins with a single, simple step.", "The doctor said I should walk for twenty minutes daily.", "They planted mango and neem trees along the school road.", "What time does the pharmacy close tonight?", "The mobile network is weak inside the old temple.", "Rice, dal, and a little pickle make a perfect lunch.", "He fixed the broken chair with glue and two small nails.", "Electric scooters are becoming popular in big cities.", "The festival lights made the whole street glow at night.", "I'll be there in fifteen minutes, so please wait.", "The cat jumped onto the wall and watched the birds.", "Learning a new language takes patience and practice.", "Monsoon clouds gathered quickly over the lake.", "Thank you very much for your kind help yesterday.", "Please check whether the gas stove is switched off.", "The price of tomatoes went up again this week.", "A gentle breeze moved through the coconut trees.", "Our team finished the project two days before the deadline.", "Can you remind me to pay the electricity bill on Friday?", "The cinema hall was full for the evening show.", "My favourite colour has always been deep blue.", "We need eggs, milk, bread, and a packet of sugar.", "The engineer explained the design in simple words.", "Shall I read your latest notifications aloud?", "Opening Google Maps with directions to Charminar.", "The timer is set for ten minutes.", "I've saved that note for you.", "Here are the top results for your search.", "Your calendar is free for the rest of the day.", "Okay, I will remember that.", "Taking a screenshot now.", "Locking the phone. See you soon.", "That's one hundred and twenty five, Boss.", "Fifteen percent of two thousand is three hundred.", "Which video would you like me to play?", "Scrolling down to show more results.", "Playing the second video now.", "Pausing the music.", "Skipping to the next song.", "I'm not sure about that, but I can search the web for you.", "Would you like me to call them back?", "The message has been sent successfully.", "Please unlock your phone first.", "Speech is a little faster now. Is this better?", "Zebras, giraffes, and elephants live on the wide savanna.", "Judge the quality of the work, not just the quantity.", "Vivid violet flowers grew beside the village well.", "The thick fog made it hard to see the highway.", "Shy sheep sleep in the shade of the shed.", "Each yellow chair was placed in a perfect row.", "Measure twice and cut once, the carpenter said.", "Jasmine has a sweet smell that fills the evening air.", "The orchestra played a joyful tune at the wedding.", "Please queue patiently at the counter for your token.", "Exactly eighty eight kilometres separate the two towns.", "Hurry up, the bus is about to leave!", "Is it going to rain later this afternoon?", "Wow, that was a wonderful surprise!", "No, I don't think that's a good idea.", "Yes, absolutely, let's do it right now.", "Why is the sky blue during the day and orange at sunset?", "How many days are left until the exam?", "Where did I keep my keys last night?", "Who scored the winning goal in the second half?", "Numbers like seven, eleven, and nineteen are prime.", "The temperature dropped to eighteen degrees at night.", "My phone number ends with four, three, two, one.", "The appointment is on the twenty second of March.", "We drove four hundred kilometres in a single day.", "The shop gave a discount of twenty percent on shoes.", "Please speak slowly and clearly into the microphone.", "Artificial intelligence helps people do everyday tasks.", "The software update will finish in a few minutes.", "Bluetooth is turned off, so the headphones aren't connected.", "Wi-Fi is connected to the home network.", "Turn left at the next signal and then go straight.", "The parcel will be delivered by tomorrow evening.", "I'd like a window seat on the morning flight, please.", "Our cricket team practices every evening after school.", "The museum has paintings from many different centuries.", "Fresh fruit juice is better than sugary soft drinks.", "The farmer checked the soil before sowing the seeds.", "Remember to back up your photos every month.", "The batteries in the remote control are almost empty.", "Grandfather reads the newspaper with his morning tea.", "The puppy chased its tail round and round in circles.", "Bright stars filled the sky above the quiet village.", "Good night, Boss. Sleep well.", "Good afternoon, Boss. You have two events today.", "Good evening, Boss. The traffic is heavy on the main road.", "Don't worry, everything is under control.", "That sounds like a great plan to me.", "I'm sorry, I can't do that on this phone.", "Let me check that for you.", "Just a moment, I'm thinking.", "Here's a quick summary of your day.", "You're welcome, Boss. Happy to help.", "Ready when you are.", "Done!", "All set, Boss.", "Hmm, let me think about that.", "Okay, cancelling that for you.", "Would you like to hear it again?"]

def save22k(y, sr, name):
    if sr != 22050: y = librosa.resample(y, orig_sr=sr, target_sr=22050)
    y = y / max(1e-6, np.abs(y).max()) * 0.9
    sf.write(f"{WAVS}/{name}.wav", y, 22050, subtype="PCM_16")

if "jarvis-voice-data.zip" in files:
    print("Using the sentences you recorded in Jarvis.")
    !unzip -o -q "{JARVIS}/jarvis-voice-data.zip" -d /content/zipdata
    for line in open("/content/zipdata/metadata.csv", encoding="utf-8"):
        if "|" not in line: continue
        uid, text = line.rstrip("\n").split("|", 1)
        y, sr = librosa.load(f"/content/zipdata/wavs/{uid}.wav", sr=None, mono=True)
        save22k(y, sr, uid); rows.append((uid, text))
elif any(f.startswith("short-clip") for f in files) and not [f for f in files if f.lower().endswith(AUDIO) and not f.startswith("short-clip")]:
    print("Quick mode: making training sentences in your voice from your short clip (about 20 minutes).")
    clip = [f for f in files if f.startswith("short-clip")][0]
    !wget -q -nc https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models/sherpa-onnx-pocket-tts-int8-2026-01-26.tar.bz2 -P /content
    !tar xjf /content/sherpa-onnx-pocket-tts-int8-2026-01-26.tar.bz2 -C /content
    import sherpa_onnx as so
    P = glob.glob("/content/sherpa-onnx-pocket-tts-int8-*/")[0]
    pc = so.OfflineTtsPocketModelConfig(lm_flow=P+"lm_flow.int8.onnx", lm_main=P+"lm_main.int8.onnx", encoder=P+"encoder.onnx", decoder=P+"decoder.int8.onnx",
                                        text_conditioner=P+"text_conditioner.onnx", vocab_json=P+"vocab.json", token_scores_json=P+"token_scores.json")
    mc = so.OfflineTtsModelConfig(num_threads=2); mc.pocket = pc
    tts = so.OfflineTts(so.OfflineTtsConfig(model=mc))
    ref, rsr = librosa.load(f"{JARVIS}/{clip}", sr=24000, mono=True)
    ref = ref[: 24000 * 60]
    g = so.GenerationConfig(); g.reference_audio = ref.tolist(); g.reference_sample_rate = 24000
    for i, text in enumerate(SENTENCES):
        a = tts.generate(text, g)
        save22k(np.array(a.samples, dtype=np.float32), a.sample_rate, f"q{i:04d}"); rows.append((f"q{i:04d}", text))
        if i % 20 == 0: print(i, "/", len(SENTENCES))
else:
    long_files = [f for f in files if f.lower().endswith(AUDIO)]
    assert long_files, "Put your recording (.mp3/.m4a/.wav) in the Jarvis folder of your Google Drive, then Run all again."
    print("Writing down what you said in:", long_files)
    from faster_whisper import WhisperModel
    wm = WhisperModel("large-v3", device="cuda", compute_type="float16")
    n = 0
    for f in long_files:
        y, sr = librosa.load(f"{JARVIS}/{f}", sr=22050, mono=True)
        segs, _ = wm.transcribe(f"{JARVIS}/{f}", language="en", vad_filter=True, beam_size=5)
        for s in segs:
            dur = s.end - s.start
            if dur < 1.0 or dur > 15 or s.no_speech_prob > 0.5 or s.avg_logprob < -1.0 or len(s.text.strip()) < 3: continue
            a, b = int(max(0, s.start - 0.1) * 22050), int(min(len(y) / 22050, s.end + 0.15) * 22050)
            save22k(y[a:b], 22050, f"r{n:05d}"); rows.append((f"r{n:05d}", s.text.strip())); n += 1
    del wm

with open(DATA + "/metadata.csv", "w", encoding="utf-8") as m:
    for uid, text in rows: m.write(f"{uid}.wav|{text}\n")
mins = sum(sf.info(f"{WAVS}/{u}.wav").duration for u, _ in rows) / 60
print(f"Ready: {len(rows)} clips, {mins:.1f} minutes of your voice.")
if mins < 10: print("Tip: more than 15 minutes sounds much better. It will still work.")

In [ ]:
#@title 4. Train your voice (2-5 hours). Safe to re-run: it continues.
CKPT_DIR = JARVIS + "/training"; os.makedirs(CKPT_DIR, exist_ok=True)
saved = CKPT_DIR + "/last.ckpt"
if os.path.exists(saved):
    start = saved; print("Continuing from your saved training.")
else:
    start = "/content/lessac.ckpt"
    if not os.path.exists(start):
        !wget -q -O /content/lessac.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt"
    print("Starting from a ready-made English voice (Lessac medium).")
MAX_EPOCHS = 2164 + EXTRA_EPOCHS
cmd = ["python", "-m", "piper.train", "fit",
       "--data.voice_name", "jarvis_me", "--data.csv_path", DATA + "/metadata.csv", "--data.audio_dir", WAVS,
       "--model.sample_rate", "22050", "--data.espeak_voice", "en-us", "--data.cache_dir", "/content/cache",
       "--data.config_path", "/content/config.json", "--data.batch_size", "16",
       "--model.mos_metric", "none", "--trainer.max_epochs", str(MAX_EPOCHS),
       "--trainer.default_root_dir", "/content/train", "--ckpt_path", start]
log = open("/content/train.log", "w")
p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd="/content/piper1-gpl")
last_copy = time.time()
def newest():
    c = glob.glob("/content/train/lightning_logs/*/checkpoints/last.ckpt")
    return max(c, key=os.path.getmtime) if c else None
while p.poll() is None:
    time.sleep(60)
    tail = open("/content/train.log").read()[-400:].replace("\r", "\n").splitlines()
    print(time.strftime("%H:%M"), tail[-1][:160] if tail else "")
    if time.time() - last_copy > 15 * 60 and newest():
        shutil.copy(newest(), saved + ".tmp"); os.replace(saved + ".tmp", saved); shutil.copy("/content/config.json", CKPT_DIR + "/config.json")
        last_copy = time.time(); print("  (progress saved to your Drive)")
if newest():
    shutil.copy(newest(), saved + ".tmp"); os.replace(saved + ".tmp", saved); shutil.copy("/content/config.json", CKPT_DIR + "/config.json")
print("Training finished with code", p.returncode, "- if it is not 0, see /content/train.log")

In [ ]:
#@title 5. Make jarvis-my-voice.zip and listen to a sample
# Converts a Piper voice (model.onnx + config.json) into the files sherpa-onnx needs.
import json, onnx, sys
def convert(onnx_in, config_in, onnx_out, tokens_out):
    cfg = json.load(open(config_in, encoding="utf-8"))
    with open(tokens_out, "w", encoding="utf-8") as f:
        for sym, ids in cfg["phoneme_id_map"].items():
            f.write(f"{sym} {ids[0]}\n")
    m = onnx.load(onnx_in)
    del m.metadata_props[:]
    meta = {"model_type": "vits", "comment": "piper",
            "language": (cfg.get("language") or {}).get("name_english", "English"),
            "voice": cfg["espeak"]["voice"], "has_espeak": 1,
            "n_speakers": cfg.get("num_speakers", 1), "sample_rate": cfg["audio"]["sample_rate"]}
    for k, v in meta.items():
        p = m.metadata_props.add(); p.key = k; p.value = str(v)
    onnx.save(m, onnx_out)
if __name__ == "__main__" and len(sys.argv) == 5 and sys.argv[1].endswith(".onnx"):
    convert(*sys.argv[1:5])

OUT = "/content/out"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
!python -m piper.train.export_onnx --checkpoint "{JARVIS}/training/last.ckpt" --output-file /content/raw.onnx
cfg = "/content/config.json" if os.path.exists("/content/config.json") else JARVIS + "/training/config.json"
convert("/content/raw.onnx", cfg, OUT + "/model.onnx", OUT + "/tokens.txt")
!wget -q -nc https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models/vits-piper-en_US-lessac-medium.tar.bz2 -P /content
!tar xjf /content/vits-piper-en_US-lessac-medium.tar.bz2 -C /content
shutil.copytree("/content/vits-piper-en_US-lessac-medium/espeak-ng-data", OUT + "/espeak-ng-data")
import sherpa_onnx as so
v = so.OfflineTtsVitsModelConfig(model=OUT + "/model.onnx", tokens=OUT + "/tokens.txt", data_dir=OUT + "/espeak-ng-data")
tts = so.OfflineTts(so.OfflineTtsConfig(model=so.OfflineTtsModelConfig(vits=v, num_threads=2)))
a = tts.generate("Hello Boss, this is Jarvis speaking in your voice. How do I sound?", sid=0, speed=1.0)
sf.write("/content/sample.wav", np.array(a.samples), a.sample_rate)
shutil.make_archive("/content/jarvis-my-voice", "zip", OUT)
shutil.copy("/content/jarvis-my-voice.zip", JARVIS + "/jarvis-my-voice.zip")
print("Done! jarvis-my-voice.zip is in your Drive's Jarvis folder. Download it to your phone, then Jarvis setup > Use my voice.")
from IPython.display import Audio; Audio("/content/sample.wav")